In [1]:
!pip install psmpy


In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from psmpy import PsmPy
from psmpy.functions import cohenD
from psmpy.plotting import *

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
save_path = '/content/drive/MyDrive/Capital_trials/'
full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')
ohe_meta = pd.read_csv('/content/drive/MyDrive/Capital_trials/ohe_metadata_only.csv')

#Create New matched Groups based on Scores

taking a caliper + NN approach


In [16]:
ohe_meta.columns

Index(['Unnamed: 0', 'Name', 'Continuing threat to society',
       'Disregard for human life', 'Manner - mayhem',
       'Multiple - harm to multiple people', 'Retaliation',
       'Victim - old or disabled', 'Weapon - arson', 'Weapon - deadly weapon',
       'aggravated assault', 'attempted rape', 'burglary/robbery',
       'child victim', 'created public risk', 'deliberate and premeditated',
       'done while committing multiplie felonies', 'especially cruel',
       'fiancial gain', 'financial gain', 'financial gian', 'gang murder',
       'intentional infliction of great bodily injury',
       'interfere with prosecution/arrest', 'kidnapping', 'larceny',
       'lewd acts with a child', 'lying in wait', 'multiple murders',
       'murdered an officer in the line of duty', 'personal use of a handgun',
       'poison', 'prior conviction', 'prior offense', 'prior offenses', 'rape',
       'robbery', 'sexual assult', 'shooting at an occupied vehicle', 'sodomy',
       'torture', 'unl

In [ ]:
print(caliper_race_crime)
print(caliper_state_race_crime)

In [17]:
df_encoded= ohe_meta

In [22]:
#implementation with psmpy

#race crime and state
X = df_encoded[['State_AL', 'State_AZ', 'State_CA','State_FL', 'State_GA',
 'State_LA', 'State_MS', 'State_NC', 'State_OH',
 'State_OK', 'State_PA', 'State_SC', 'State_TN', 'State_TX',
  'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]]
T = df_encoded['gender_women']

psm_df =  X.copy()
psm_df['gender'] = T.astype(int)

psm_df = psm_df.reset_index(drop=True)
psm_df["row_id"] = psm_df.index

psm = PsmPy(psm_df, treatment="gender", indx="row_id", exclude=[])
psm.logistic_ps(balance=True)
pred = psm.predicted_data
print(pred.columns)

full_meta = full_meta.reset_index(drop=True)
full_meta["race_crime_state_prop_scores"] = pred["propensity_score"].values

psm.kdtree_matched(matcher="propensity_score", replacement=False, caliper=None)
matched_df = psm.df_matched
matched_ids = set(psm.df_matched["row_id"].unique())
full_meta = full_meta.reset_index(drop=True)
full_meta["psmpropensity_state_race_crime"] = full_meta.index.isin(matched_ids)





Index(['row_id', 'State_AL', 'State_AZ', 'State_CA', 'State_FL', 'State_GA',
       'State_LA', 'State_MS', 'State_NC', 'State_OH', 'State_OK', 'State_PA',
       'State_SC', 'State_TN', 'State_TX', 'Race/Ethnicity_Asian',
       'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
       'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
       'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
       'Continuing threat to society', 'Disregard for human life',
       'Manner - mayhem', 'Multiple - harm to multiple people', 'Retaliation',
       'Victim - old or disabled', 'Weapon - arson', 'Weapon - deadly weapon',
       'aggravated assault', 'attempted rape', 'burglary/robbery',
       'child victim', 'created public risk', 'deliberate and premeditated',
       'done while committing multiplie felonies', 'especially cruel',
       'fiancial gain', 'financial gain', 'financial gian', 'gang murder',
       'intentional infliction of great bodily injury',
       

In [23]:
full_meta

,Unnamed: 0,document_name,Name,State,Race/Ethnicity,gender,race_crime_state_prop_scores,psmpropensity_state_race_crime
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,AL,White,women,0.756653,True
1,1,Heather Keaton.txt,Heather Leavell Keaton,AL,"Native American,White,Black",women,0.804618,True
2,2,Lisa Graham.txt,Lisa Carpenter Graham,AL,White,women,0.639617,True
3,3,Patricia Blackmon.txt,Patricia Blackmon,AL,Black,women,0.635082,True
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,AL,White,women,0.713026,True
...,...,...,...,...,...,...,...,...
100,100,Manuel Velez Combined Transcript.txt,"Velez, Manuel",TX,Latino,men,0.043996,False
101,101,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",TX,Black,men,0.180098,False
102,102,Perry Williams Combined Transcript.txt,"Williams, Perry E",TX,Black,men,0.180098,False
103,103,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",TX,White,men,0.228538,True


In [25]:
X = df_encoded[['Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]]
T = df_encoded['gender_women']


psm_df =  X.copy()
psm_df['gender'] = T.astype(int)

psm_df = psm_df.reset_index(drop=True)
psm_df["row_id"] = psm_df.index

psm = PsmPy(psm_df, treatment="gender", indx="row_id", exclude=[])
psm.logistic_ps(balance=True)
pred = psm.predicted_data
full_meta = full_meta.reset_index(drop=True)
full_meta["race_crime_prop"] = pred["propensity_score"].values

psm.kdtree_matched(matcher="propensity_score", replacement=False, caliper=None)
matched_df = psm.df_matched
matched_ids = set(psm.df_matched["row_id"].unique())
full_meta = full_meta.reset_index(drop=True)
full_meta["psmpropensity_race_crime"] = full_meta.index.isin(matched_ids)

In [26]:
full_meta

,Unnamed: 0,document_name,Name,State,Race/Ethnicity,gender,race_crime_state_prop_scores,psmpropensity_state_race_crime,race_crime_prop,psmpropensity_race_crime
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,AL,White,women,0.756653,True,0.764350,True
1,1,Heather Keaton.txt,Heather Leavell Keaton,AL,"Native American,White,Black",women,0.804618,True,0.792465,True
2,2,Lisa Graham.txt,Lisa Carpenter Graham,AL,White,women,0.639617,True,0.624890,True
3,3,Patricia Blackmon.txt,Patricia Blackmon,AL,Black,women,0.635082,True,0.565850,True
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,AL,White,women,0.713026,True,0.700201,True
...,...,...,...,...,...,...,...,...,...,...
100,100,Manuel Velez Combined Transcript.txt,"Velez, Manuel",TX,Latino,men,0.043996,False,0.069327,False
101,101,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",TX,Black,men,0.180098,False,0.374027,True
102,102,Perry Williams Combined Transcript.txt,"Williams, Perry E",TX,Black,men,0.180098,False,0.374027,True
103,103,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",TX,White,men,0.228538,True,0.384603,True


In [27]:
full_meta.to_csv(save_path+ '/meta_with_psm_matches.csv')